In [4]:
import pandas as pd
import numpy as np

# ---------------------------
# Load dataset
# ---------------------------
data = pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\Stu Activity.csv")
print("Original dataset shape:", data.shape)
print(data.isnull().sum())

# ---------------------------
# 1. Remove duplicates
# ---------------------------
data = data.drop_duplicates()

# ---------------------------
# 2. Handle missing values
# ---------------------------

# Separate numerical and categorical columns
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = data.select_dtypes(include=['object']).columns

# Fill numerical columns with median
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# Fill categorical columns with mode
for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# ---------------------------
# 3. Standardize categorical/text data
# ---------------------------
for col in cat_cols:
    data[col] = data[col].str.lower().str.strip()

# ---------------------------
# 4. Remove invalid entries
# ---------------------------

# Example numeric validation
if 'CGPA' in data.columns:
    data = data[(data['CGPA'] >= 0) & (data['CGPA'] <= 10)]
if 'IQ' in data.columns:
    data = data[data['IQ'] > 0]
if 'Attendance' in data.columns:
    data = data[(data['Attendance'] >= 0) & (data['Attendance'] <= 100)]

# ---------------------------
# 5. Handle outliers using IQR
# ---------------------------
def cap_outliers(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    col = np.where(col < lower, lower, col)
    col = np.where(col > upper, upper, col)
    return col

for col in num_cols:
    data[col] = cap_outliers(data[col])

# ---------------------------
# 6. Summary of changes
# ---------------------------
summary = {
    "Initial shape": pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\Stu Activity.csv").shape,
    "After duplicates removed": data.shape,
    "Columns filled missing": list(num_cols) + list(cat_cols),
    "Columns standardized": list(cat_cols),
    "Invalid entries removed/capped": list(num_cols),
}

# ---------------------------
# 7. Save cleaned data
# ---------------------------
data.to_csv(r"C:\Users\sadvika\Desktop\Data mining\ cleaned Stu Actvity.csv", index=False)

# ---------------------------
# 8. Outputs
# ---------------------------
print("Cleaned dataset shape:", data.shape)
print("Summary of changes:", summary)


Original dataset shape: (1000, 11)
Student_ID                 0
College_ID                 0
IQ                        13
Prev_Sem_Result           10
CGPA                       9
Academic_Performance      14
Internship_Experience     10
Extra_Curricular_Score    11
Communication_Skills       9
Projects_Completed         5
Placement                 12
dtype: int64
Cleaned dataset shape: (980, 11)
Summary of changes: {'Initial shape': (1000, 11), 'After duplicates removed': (980, 11), 'Columns filled missing': ['Student_ID', 'IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance', 'Extra_Curricular_Score', 'Communication_Skills', 'Projects_Completed', 'College_ID', 'Internship_Experience', 'Placement'], 'Columns standardized': ['College_ID', 'Internship_Experience', 'Placement'], 'Invalid entries removed/capped': ['Student_ID', 'IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance', 'Extra_Curricular_Score', 'Communication_Skills', 'Projects_Completed']}


In [14]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\Stu Performance.csv")

# ---------------------------
# 1. Display Original Dataset Info
# ---------------------------
print("Original Dataset Shape:", df.shape)
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])
print("Total Values in Dataset:", df.size)

# ---------------------------
# 2. Display Missing Values
# ---------------------------
print("\nMissing Values in Each Column:")
print(df.isnull().sum())

print("\nTotal Missing Values in Dataset:", df.isnull().sum().sum())

summary = {}


# 1. Remove duplicates
initial_rows = len(df)
df = df.drop_duplicates()
summary["duplicates_removed"] = initial_rows - len(df)

# 2. Fix invalid Exam_Score values
invalid_exam = df[(df["Exam_Score"] > 100) | (df["Exam_Score"] < 0)].shape[0]
df.loc[df["Exam_Score"] > 100, "Exam_Score"] = 100
df.loc[df["Exam_Score"] < 0, "Exam_Score"] = 0
summary["invalid_exam_scores_fixed"] = invalid_exam

# 3. Standardize categorical columns
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].str.strip().str.title()
summary["categorical_columns_standardized"] = list(cat_cols)

# 4. Handle outliers using IQR
num_cols = df.select_dtypes(include=np.number).columns
outliers_fixed = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    if count > 0:
        outliers_fixed[col] = int(count)

summary["outliers_capped"] = outliers_fixed

# Save cleaned dataset
df.to_csv(r"C:\Users\sadvika\Desktop\Data mining\ cleaned Stu Performance.csv", index=False)
print("Cleaned dataset shape:", data.shape)
print("Summary of changes:", summary)

Original Dataset Shape: (1000, 22)
Total Rows: 1000
Total Columns: 22
Total Values in Dataset: 22000

Missing Values in Each Column:
Student_ID                     0
Hours_Studied                 11
Attendance                    11
Parental_Involvement          16
Access_to_Resources           10
Extracurricular_Activities    13
Sleep_Hours                    9
Previous_Scores               14
Motivation_Level               9
Internet_Access                9
Tutoring_Sessions             17
Family_Income                 12
Teacher_Quality               24
School_Type                   11
Peer_Influence                17
Physical_Activity             12
Learning_Disabilities         10
Parental_Education_Level      28
Distance_from_Home            27
Gender                         8
Exam_Score                     9
Academic_Performance           8
dtype: int64

Total Missing Values in Dataset: 285
Cleaned dataset shape: (980, 11)
Summary of changes: {'duplicates_removed': 0, 'invalid_ex

In [15]:
import pandas as pd

# ---------------------------
# 1. Load datasets
# ---------------------------
activity = pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\ cleaned Stu Actvity.csv")
performance = pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\ cleaned Stu Performance.csv")

# ---------------------------
# 2. Verify primary keys
# ---------------------------
assert "Student_ID" in activity.columns
assert "Student_ID" in performance.columns
assert "Academic_Performance" in activity.columns
assert "Academic_Performance" in performance.columns

# ---------------------------
# 3. Standardize key column formats
# ---------------------------
activity["Student_ID"] = activity["Student_ID"].astype(int)
performance["Student_ID"] = performance["Student_ID"].astype(int)

activity["Academic_Performance"] = activity["Academic_Performance"].astype(str).str.strip()
performance["Academic_Performance"] = performance["Academic_Performance"].astype(str).str.strip()

# ---------------------------
# 4. Merge datasets using two keys
# ---------------------------
integrated_data = pd.merge(
    activity,
    performance,
    on=["Student_ID", "Academic_Performance"],
    how="inner",
    suffixes=("_activity", "_performance")
)

# ---------------------------
# 5. Remove duplicate columns
# ---------------------------
integrated_data = integrated_data.loc[:, ~integrated_data.columns.duplicated()]

# ---------------------------
# 6. Remove duplicate records based on both keys
# ---------------------------
integrated_data.drop_duplicates(subset=["Student_ID", "Academic_Performance"], inplace=True)

# ---------------------------
# 7. Save integrated dataset
# ---------------------------
integrated_data.to_csv(r"C:\Users\sadvika\Desktop\Data mining\integrated_student_data.csv", index=False)

# ---------------------------
# 8. Output info
# ---------------------------
print("Integrated dataset shape:", integrated_data.shape)
print("Columns:", integrated_data.columns.tolist())

Integrated dataset shape: (959, 31)
Columns: ['Student_ID', 'College_ID', 'IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance', 'Internship_Experience', 'Extra_Curricular_Score', 'Communication_Skills', 'Projects_Completed', 'Placement', 'Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender', 'Exam_Score']


In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------------------------
# 1. Load dataset
# ---------------------------
data = pd.read_csv(r"C:\Users\sadvika\Desktop\Data mining\integrated_student_data.csv")

# ---------------------------
# 2. Data type conversions
# ---------------------------
numeric_cols = ["CGPA", "IQ", "Attendance", "Exam_Score"]
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

categorical_cols = ["Gender", "School_Type", "Internship_Experience", "Placement"]
for col in categorical_cols:
    if col in data.columns:
        data[col] = data[col].astype("category")

# ---------------------------
# 3. Binary encoding
# ---------------------------
binary_map = {"yes": 1, "no": 0}

if "Internship_Experience" in data.columns:
    data["Internship_Experience"] = (
        data["Internship_Experience"]
        .str.lower()
        .map(binary_map)
    )

if "Placement" in data.columns:
    data["Placement"] = (
        data["Placement"]
        .str.lower()
        .map(binary_map)
    )

# ---------------------------
# 4. One-hot encoding
# ---------------------------
data = pd.get_dummies(
    data,
    columns=["Gender", "School_Type"],
    drop_first=True
)

# ---------------------------
# 5. Feature Engineering
# ---------------------------
# Academic performance index
data["Performance_Index"] = (
    (data["CGPA"] * 0.5) +
    (data["Exam_Score"] * 0.3) +
    (data["Attendance"] * 0.2)
)

# ---------------------------
# 6. Scaling numeric features
# ---------------------------
scaler = StandardScaler()
scale_cols = ["CGPA", "IQ", "Attendance", "Exam_Score", "Performance_Index"]

data[scale_cols] = scaler.fit_transform(data[scale_cols])

# ---------------------------
# 7. Save transformed dataset
# ---------------------------
data.to_csv(r"C:\Users\sadvika\Desktop\Data mining\transformed_student_data.csv", index=False)

# ---------------------------
# 8. Output info
# ---------------------------
print("Transformed dataset shape:", data.shape)
print("Columns:", data.columns.tolist())


Transformed dataset shape: (959, 32)
Columns: ['Student_ID', 'College_ID', 'IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance', 'Internship_Experience', 'Extra_Curricular_Score', 'Communication_Skills', 'Projects_Completed', 'Placement', 'Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Exam_Score', 'Gender_Male', 'School_Type_Public', 'Performance_Index']
